# The workshop model, in three exercises

One region, hour by hour, twenty years. Plant is dispatched and the price is set
by the last unit needed. Investors decide what to build against a view of the
future they extrapolate rather than forecast, contracts move risk between them,
and a procurement scheme can be switched on.

**Everything here is illustrative, and not a forecast of anything.** The
fleet is stylised, the weather is synthetic, and the system is one region
with no transmission in it.
Each exercise says what should move and what should not. If something moves that
was supposed to stay still, that is the interesting outcome, not a mistake to
hide.

---

The runs below use a nine-cell lattice rather than the full forty-five so that a
cell finishes while you are still looking at it. That changes how precisely the
forward view is resolved. It does not change any of the orderings, coincidences
or conservation laws these exercises are about.


In [ ]:
!pip install -q esem-sandbox  # skip this line if you already have it

from esem_sandbox.config import load_settings
from esem_sandbox.core.forward import cell_plan
from esem_sandbox.core.simulate import run, MERCHANT, ESEM
from esem_sandbox import plots
import numpy as np

settings = load_settings()
FAST = tuple(c for c in cell_plan(settings) if c.shape_year == 0)
TICKS, SEED = 8, 20260904
print(f'{len(cell_plan(settings))} cells in the full lattice, {len(FAST)} in the fast one')


## Exercise 1. The same weather, with and without the scheme

Both legs draw one weather sequence from one seed, so the difference between them
is the mechanism and nothing else. A leg that drew its own weather would report
the difference between two climates as the effect of a policy.

**Should move:** capacity, unserved energy, the levy, and the bill.

**Should not move:** the weather, the growth path, or any year before the plant
the scheme awarded has been built.


In [ ]:
legs = {leg: run(settings, ticks=TICKS, seed=SEED, cells=FAST, leg=leg)
        for leg in (MERCHANT, ESEM)}

assert legs[MERCHANT].draw == legs[ESEM].draw   # one weather sequence

standard = settings.reliability['standard_use_fraction']
print(f"{'year':>6}{'merchant':>16}{'with the scheme':>18}{'lane MW':>10}")
for a, b in zip(legs[MERCHANT].ticks, legs[ESEM].ticks):
    print(f'{a.year:>6}{a.unserved_fraction/standard:>13.2f}x'
          f'{b.unserved_fraction/standard:>17.2f}x{b.lane_volume_mw:>10.0f}')


Look at the first two or three years. **The legs are identical**, however much
the lane bought in year one, because the plant it paid for has not been built
yet. A procurement scheme is an instrument about the future. It cannot fix a year
that arrives before its plant does, and that is a lead time rather than a flaw.


In [ ]:
plots.dashboard(legs, settings, 'dashboard.png')
from IPython.display import Image
Image('dashboard.png')


### The bill is not the cost

Most of a bill is a payment from consumers to producers. A scheme that builds
capacity pushes the pool price down and cuts the bill by far more than it costs,
and that reduction is a **transfer**, not a saving. Read both lines or the
comparison will report the transfer as a benefit.


In [ ]:
m, e = legs[MERCHANT], legs[ESEM]
bill = m.consumer_cost(settings) - e.consumer_cost(settings)
real = m.resource_cost(settings) - e.resource_cost(settings)
print(f'the bill moves          {bill/1e9:>8,.2f} bn')
print(f'the resource cost moves {real/1e9:>8,.2f} bn')
print(f'the difference is a transfer of {(bill-real)/1e9:,.2f} bn')


**Now change the seed.** Run the cell below with 20260101, then 19990101. What
survives a different weather draw, and what was a property of this one?


In [ ]:
for seed in (20260904, 20260101, 19990101):
    pair = {leg: run(settings, ticks=TICKS, seed=seed, cells=FAST, leg=leg)
            for leg in (MERCHANT, ESEM)}
    a, b = pair[MERCHANT], pair[ESEM]
    print(f'seed {seed}  {a.draw.growth_path:>7} growth   '
          f'unserved {a.total_unserved_gwh:>7.1f} -> {b.total_unserved_gwh:>7.1f} GWh'
          f'   resource cost moves '
          f'{(a.resource_cost(settings)-b.resource_cost(settings))/1e9:>6,.2f} bn')


## Exercise 2. The decomposition dial

The scheme reaches the market through three channels, and the tempting story
attributes the whole effect to the first one. Each switch is here so you can see
what it is actually worth.

1. **Exposure.** A long contract removes spot-price risk, so the certainty
   equivalent of a plant's rent rises toward its expected value.
2. **The cost of capital.** A contracted megawatt is financed as debt would be
   and an uncontracted one as equity is.
3. **The quantity procured.** The lane simply buys plant.

**Should move:** what a cap costs, what an award costs, and the fleet.

**Should not move:** anything at all, when all three are off.


In [ ]:
# Switch three: nothing procured. With a standard nothing can breach, the lane
# is empty in every year, and the scheme leg must be the merchant leg exactly.
shut = load_settings({'reliability': {'standard_use_fraction': 1.0}})
a = run(shut, ticks=TICKS, seed=SEED, cells=FAST, leg=MERCHANT)
b = run(shut, ticks=TICKS, seed=SEED, cells=FAST, leg=ESEM)

same = all(x.unserved_gwh == y.unserved_gwh and x.mean_price == y.mean_price
           for x, y in zip(a.ticks, b.ticks))
print('the two legs coincide exactly:', same)
print('lane opened in any year:', any(t.lane_volume_mw for t in b.ticks))


In [ ]:
# Switch one: risk aversion off. The certainty equivalent becomes the expected
# value and every hurdle becomes its plain fixed cost.
flat = load_settings({'investment': {'risk_premium': 0.0}})
base = run(settings, ticks=TICKS, seed=SEED, cells=FAST)
calm = run(flat, ticks=TICKS, seed=SEED, cells=FAST)

built = lambda r: sorted((b.technology, b.capacity_mw) for t in r.ticks for b in t.builds)
print('the fleet changes:', built(base) != built(calm))
caps = lambda r: max(c.premium_per_mwh for c in r.book if c.kind == 'cap')
print(f'a cap costs {caps(base):.2f} against {caps(calm):.2f} $/MWh')


**Risk aversion alone does not close the gap.** It changes what insurance costs,
because a cap is priced on the same coefficient. On this fleet it moves a
peaker's hurdle by a few per cent of its fixed cost, which is real and is not
enough to flip a build. Anyone who tells you the whole effect of a long contract
is the risk it removes is describing a different model.


In [ ]:
# Switch two: the financing advantage removed, by contracting at the merchant rate.
level = load_settings({'esem': {'contracted_wacc': settings.tech('ocgt').wacc}})
rich = run(settings, ticks=4, seed=SEED, cells=FAST, leg=ESEM)
poor = run(level, ticks=4, seed=SEED, cells=FAST, leg=ESEM)
print(f'awards cost {sum(t.scheme_cost for t in rich.ticks)/1e6:>10,.1f} m'
      f' at the contracted rate')
print(f'            {sum(t.scheme_cost for t in poor.ticks)/1e6:>10,.1f} m'
      f' at the merchant rate')


## Exercise 3. Why a duration curve, and why the hourly series

This model settles every contract on the full 8,760-hour series and never on a
sampled curve or a block average. This exercise is why.

**Should move:** the cap payout, enormously.

**Should not move:** the price series itself. It is the same year either way.


In [ ]:
from esem_sandbox.core.dispatch import dispatch_year
from esem_sandbox.core.weather import generate_bundle
from esem_sandbox.core.report import quarter_of_hour

bundle = generate_bundle(settings.weather['seed'], settings.weather['shape_years'])
shape = bundle['demand_shape'][4]          # the lull-on-heat year
year = dispatch_year(settings, 2026, shape * (12500 / shape.max()),
                     bundle['wind_cf'][4], bundle['solar_cf'][4])

K = settings.contracts['cap_strike_per_mwh']
hourly = np.clip(year.price - K, 0, None).sum()
quarters = quarter_of_hour(len(year.price))
on_means = sum(max(year.price[quarters == q].mean() - K, 0) * (quarters == q).sum()
               for q in range(4))
print(f'a $  {K:.0f} cap, settled hour by hour:      {hourly:>12,.0f} $/MW-year')
print(f'the same cap, settled on quarterly means: {on_means:>12,.0f} $/MW-year')
print(f'the averaging destroys {1 - on_means/hourly:.1%} of the payout')


That gap is Jensen's inequality, and it is the whole reason a cap is written on
an integral. Averaging first and taking the positive part afterwards prices
insurance against a year that never happened.


In [ ]:
from esem_sandbox.core.windows import locate_worst_window
window = locate_worst_window(year.residual_mw, year.firm_capacity_mw)
plots.worst_week(year, window, year.firm_capacity_mw, 'worst_week.png')
print(f'the worst window found: {window.days} days from day {window.start_day}')
Image('worst_week.png')


In [ ]:
# What the stores actually delivered in the hours load was shed.
shed = year.unserved_mwh > 0
stores = [u for u in settings.fleet if u.technology in ('battery', 'phes')]
rated = sum(u.available_mw for u in stores) * shed.sum()
got = sum(year.generation_mwh[u.unit] for u in stores)[shed].sum()
print(f'{int(shed.sum())} hours of load shedding')
print(f'storage delivered {got:,.0f} of {rated:,.0f} MWh of rated power ({got/rated:.0%})')


## Homework

Each of these is one changed line.

- **Tenor.** `{'esem': {'contract_tenor_years': 6}}`, then `0`. Tenor is the
  channel a long contract works through, not the fraction of output it covers.
- **Boom and bust.** Pull a coal retirement forward in `fleet.csv` against the
  two-year gas lead and watch entry cluster, overshoot, and stop.
- **The strike, and the administrator's conduct.**
  `{'esem': {'recycling_conduct': 'fire_sale'}}`.
- **One year by hand.** Print the offer stack, the sorted prices, the hours at
  each rung of the demand-response ladder, and the megawatt hours at the price
  cap. Everything in this model is that arithmetic repeated.

## A glossary, for reading the larger model afterwards

| here | there |
|---|---|
| lane | the product an auction clears, one per contract type |
| tranche | one delivery year of a lane, offered on its own |
| exposure | the share of a project's life still facing the spot price |
| capture | the price a plant actually receives, given when it runs |
| the located window | the worst contiguous run of days, found rather than authored |
| the lattice | the enumerated set of futures and the weights on them |
